In [ ]:
"""
벡터 데이터베이스 - 유사 이미지 검색 : colab에서 실습
원본 이미지 URL : https://www.kaggle.com/trolukovich/datasets
캐글 데이터의 일부만 이용해 실습 : https://github.com/kairess/toy-datasets

이번 예제는 이런 구조
이미지
     → AutoImageProcessor로 전처리
     → ViTModel로 이미지 특징 벡터 추출
     → ChromaDB에 저장
     → 새 이미지 벡터와 비교
     → 유사 이미지 검색
 즉, 이미지로 이미지를 찾는 예제.

CLIP이 아니라 ViT/DINO 모델을 사용한 이미지 임베딩 예제다.
텍스트로 이미지를 검색하려면 CLIP처럼 이미지 인코더와 텍스트 인코더를 함께 가진 모델이 필요하다.

# 3개의 함수
 image_to_embedding() : 이미지를 벡터로 변환
 collection.upsert() : 벡터를 ChromaDB에 저장
 queryFunc() : 새 이미지와 유사한 이미지 검색
"""

In [ ]:
!pip install -q chromadb transformers

!wget -q https://github.com/kairess/toy-datasets/raw/master/Food-11.zip
!unzip -q Food-11.zip


In [ ]:
import torch
import chromadb
import requests
import matplotlib.pyplot as plt
from PIL import Image
from glob import glob
from tqdm import tqdm
from transformers import AutoImageProcessor, ViTModel
# AutoImageProcessor : 이미지를 모델에 넣기 전, 모델이 이해할 수 있는 형태로 바꿔주는 도구

# 모델 준비
model_name = "facebook/dino-vits16"
image_processor = AutoImageProcessor.from_pretrained(model_name)  # 이미지 전처리 도구를 불러온다.
model = ViTModel.from_pretrained(model_name)  # ViT 모델을 불러온다.

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()
print("Models loaded!")
print("사용 장치:", device)

# 이미지 벡터 추출 함수
def image_to_embedding(img):
    img = img.convert("RGB")

    inputs = image_processor(
        images=img,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}  # 입력 텐서를 모델과 같은 장치로 이동

    with torch.no_grad():
        outputs = model(**inputs)  # ViT 모델로 이미지 특징을 추출

    embedding = outputs.last_hidden_state[:, 0, :]  # 첫 번째 토큰인 CLS 토큰을 이미지 대표 벡터로 사용

    embedding = embedding / embedding.norm(
        p=2,
        dim=-1,
        keepdim=True
    )

    return embedding.squeeze(0).cpu().numpy().tolist()  # ChromaDB 저장용 list로 반환


client = chromadb.Client()  # 메모리 기반 ChromaDB 클라이언트를 생성

try:
    client.delete_collection("foods")
except:
    pass

collection = client.create_collection("foods")  # foods 컬렉션
print("ChromaDB 컬렉션 준비 완료")

img_list = sorted(glob("test/*/*.jpg"))  # test 하위의 모든 서브폴더 안에 있는 jpg 파일을 가져온다.
print("전체 이미지 수:", len(img_list))

embeddings = []  # 이미지 벡터를 저장할 리스트
metadatas = []    # 이미지 경로와 클래스명을 저장할 리스트
ids = []           # ChromaDB에 저장할 고유 ID 리스트

for i, img_path in enumerate(tqdm(img_list)):
    img = Image.open(img_path)
    cls = img_path.split("/")[1]  # 이미지 경로에서 클래스 이름을 추출
    embedding = image_to_embedding(img)  # 이미지를 ViT 벡터로 변환

    embeddings.append(embedding)
    metadatas.append({"uri": img_path, "name": cls})
    ids.append(str(i))

print("벡터화 완료:", len(embeddings))

# ChromaDB에 저장
collection.upsert(
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

# URL 이미지로 유사 이미지 검색 함수
def queryFunc(img_url, n_results=3):
    test_img = Image.open( requests.get(img_url, stream=True).raw).convert("RGB")

    test_embedding = image_to_embedding(test_img)  # 테스트 이미지를 벡터로 변환
    query_result = collection.query(
        query_embeddings=[test_embedding],
        n_results=n_results
    )
    fig, axes = plt.subplots(1,  n_results + 1,  figsize=(16, 5))

    axes[0].imshow(test_img)
    axes[0].set_title("Query")
    axes[0].axis("off")

    for i, metadata in enumerate(query_result["metadatas"][0]):
        distance = query_result["distances"][0][i]

        result_img = Image.open(metadata["uri"])

        axes[i + 1].imshow(result_img)
        axes[i + 1].set_title(f"{metadata['name']}: {distance:.2f}")
        axes[i + 1].axis("off")

    plt.show()

    return query_result  # 검색 결과를 반환


In [ ]:
# 9. 테스트
queryFunc("https://i.ibb.co/JmpXmvx/QCado9g.jpg")  # 첫 번째 테스트 이미지를 검색
queryFunc("https://i.ibb.co/X5dkHGF/lf5C0LI.png")  # 두 번째 테스트 이미지를 검색
